In [293]:
# ============================================================================
# Standard Library Imports
# ============================================================================
import pprint
from typing import Dict

# ============================================================================
# Data Analysis Library Imports
# ============================================================================
import pandas as pd
import numpy as np

# ============================================================================
# Data Visualization Library Imports
# ============================================================================
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================================
# Scikit-Learn Model Selection Imports
# ============================================================================
from sklearn.model_selection import train_test_split, RandomizedSearchCV

# ============================================================================
# Scikit-Learn Pipeline-related Imports
# ============================================================================
from sklearn.pipeline import Pipeline

# ============================================================================
# Scikit-Learn Preprocessing Imports
# ============================================================================
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# ============================================================================
# Scikit-Learn Metrics Imports
# ============================================================================
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# ============================================================================
# Scikit-Learn Model Imports
# ============================================================================
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier  

# ============================================================================
# Third-party Model Imports
# ============================================================================
from catboost import CatBoostClassifier


# ============================================================================
# Constants and Configuration
# ============================================================================
RANDOM_STATE = 1776
TEST_SIZE = 0.2

In [294]:
df = pd.read_csv(filepath_or_buffer="MBA.csv")

- `application_id`: Unique identifier for each application

- `gender`: Applicant's gender (Male, Female)

- `international`: International student (TRUE/FALSE)

- `gpa`: Grade Point Average of the applicant (on 4.0 scale)

- `major`: Undergraduate major (Business, STEM, Humanities)

- `race`: Racial background of the applicant (e.g., White, Black, Asian, Hispanic, Other / null: international student)

- `gmat`: GMAT score of the applicant (800 points)

- `work_exp`: Number of years of work experience (Year)

- `work_industry`: Industry of the applicant's previous work experience (e.g., Consulting, Finance, Technology, etc.)

- `admission`: Admission status (Admit, Waitlist, Null: Deny)

### Data Preparation

In [295]:
df.shape

(6194, 10)

In [296]:
df["application_id"].unique()

array([   1,    2,    3, ..., 6192, 6193, 6194], shape=(6194,))

In [297]:
df = df.drop(columns=["application_id"])

In [298]:
df["gender"].value_counts().index

Index(['Male', 'Female'], dtype='str', name='gender')

In [299]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6194 entries, 0 to 6193
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   gender         6194 non-null   str    
 1   international  6194 non-null   bool   
 2   gpa            6194 non-null   float64
 3   major          6194 non-null   str    
 4   race           4352 non-null   str    
 5   gmat           6194 non-null   float64
 6   work_exp       6194 non-null   float64
 7   work_industry  6194 non-null   str    
 8   admission      1000 non-null   str    
dtypes: bool(1), float64(3), str(5)
memory usage: 393.3 KB


In [300]:
df["admission"].value_counts()

admission
Admit       900
Waitlist    100
Name: count, dtype: int64

In [301]:
df["race"] = df["race"].fillna(value="unknown")
df["admission"] = df["admission"].fillna("deny")

In [302]:
df.select_dtypes(include="str")

,gender,major,race,work_industry,admission
0,Female,Business,Asian,Financial Services,Admit
1,Male,Humanities,Black,Investment Management,deny
2,Female,Business,unknown,Technology,Admit
3,Male,STEM,Black,Technology,deny
4,Male,STEM,Hispanic,Consulting,deny
...,...,...,...,...,...
6189,Male,Business,White,Other,deny
6190,Male,STEM,Black,Consulting,deny
6191,Female,Business,unknown,Health Care,Admit
6192,Male,Business,unknown,Other,deny


In [303]:
df["international"] = df["international"].apply(func=lambda status: int(bool(status)))


for column in df.select_dtypes(include="str").columns:
    df[column] = df[column].apply(func=lambda major: major.lower())

In [304]:
df.head()

,gender,international,gpa,major,race,gmat,work_exp,work_industry,admission
0,female,0,3.30,business,asian,620.0,3.0,financial services,admit
1,male,0,3.28,humanities,black,680.0,5.0,investment management,deny
2,female,1,3.30,business,unknown,710.0,5.0,technology,admit
3,male,0,3.47,stem,black,690.0,6.0,technology,deny
4,male,0,3.35,stem,hispanic,590.0,5.0,consulting,deny


In [305]:
NUMBER_OF_FEATURES = len(df.columns)
print(f"NUMBER OF FEATURES IN DATASET: {NUMBER_OF_FEATURES}")

NUMBER OF FEATURES IN DATASET: 9


It seems to me that there will be no optimzation part, because number of features is quite low

In [306]:
def print_basic_metrics(
    model_name: str, 
    y_test, y_pred, 
    include_classification_report: bool = True,
) -> None:
    """
    Print comprehensive classification performance metrics for a classification model.
    
    This function calculates and displays key evaluation metrics including accuracy,
    precision, recall, F1-score, confusion matrix, and a detailed classification report.
    All metrics are printed in a formatted, readable output.
    
    Parameters
    ----------
    model_name : str
        The name or identifier of the model being evaluated. This will be displayed
        in uppercase in the output header.
    y_test : array-like of shape (n_samples,)
        True labels for the test dataset. Must be a 1D array or list containing
        the ground truth class labels.
    y_pred : array-like of shape (n_samples,)
        Predicted labels from the model. Must be a 1D array or list containing
        the predicted class labels. Should have the same length as y_test.
    
    Returns
    -------
    None
        This function prints the metrics directly to the console and does not
        return any value.
    
    Notes
    -----
    - Precision, recall, and F1-score are calculated using weighted averaging
      to account for class imbalance.
    - The confusion matrix is printed with explicit labels for each cell
      (TN, FP, FN, TP) for easy interpretation.
    - A complete classification report from scikit-learn is also displayed,
      showing per-class metrics.
    
    Examples
    --------
    >>> from sklearn.datasets import make_classification
    >>> from sklearn.model_selection import train_test_split
    >>> from sklearn.ensemble import RandomForestClassifier
    >>> 
    >>> X, y = make_classification(n_samples=100, random_state=42)
    >>> X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
    >>> model = RandomForestClassifier(random_state=42)
    >>> model.fit(X_train, y_train)
    >>> y_pred = model.predict(X_test)
    >>> 
    >>> print_basic_metrics("Random Forest", y_test, y_pred)
    ==================================================
    RANDOM FOREST PERFORMANCE METRICS
    ==================================================
    Accuracy:                0.920
    Precision (weighted):    0.921
    Recall (weighted):       0.920
    F1-Score (weighted):     0.920
    ==================================================
    
    Confusion Matrix:
    -----------------
    True Negatives:  12
    False Positives: 1
    False Negatives: 1
    True Positives:  11
    ==================================================
    
    Classification Report:
                  precision    recall  f1-score   support
    
               0       0.92      0.92      0.92        13
               1       0.92      0.92      0.92        12
    
        accuracy                           0.92        25
       macro avg       0.92      0.92      0.92        25
    weighted avg       0.92      0.92      0.92        25
    """
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    cm = confusion_matrix(y_test, y_pred)

    print("="*50)
    print(f"{model_name.upper()} PERFORMANCE METRICS")
    print("="*50)
    print(f"Accuracy:                {accuracy:.3f}")
    print(f"Precision (weighted):    {precision:.3f}")
    print(f"Recall (weighted):       {recall:.3f}")
    print(f"F1-Score (weighted):     {f1:.3f}")
    print("="*50)

    print("\nConfusion Matrix:")
    print("-----------------")
    print(f"True Negatives:  {cm[0,0]}")
    print(f"False Positives: {cm[0,1]}")
    print(f"False Negatives: {cm[1,0]}")
    print(f"True Positives:  {cm[1,1]}")
    print("="*50)

    if include_classification_report:
      print("\nClassification Report:")
      print(classification_report(y_test, y_pred, zero_division=0))

In [307]:
classifier_metrics: Dict[str, dict] = dict()

### Logistic Regression

In [308]:
LOGREGRESSION_MODEL_NAME = "Logistic Regression"

In [309]:
numerical_features = (
    df
    .drop(columns=["admission"])
    .select_dtypes(include=np.number)
    .columns
)

categorical_features = (
	df
	.drop(columns=["admission"])
	.select_dtypes(include=["string", "object", "str"])
	.columns
)

X = df.drop(columns=["admission"])
y = df["admission"]

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE  
)

In [310]:
print(f"Categorical features are: {categorical_features}. Their length is {len(categorical_features)}")
print(f"Numerical features are: {numerical_features}. Their length is {len(numerical_features)}")

Categorical features are: Index(['gender', 'major', 'race', 'work_industry'], dtype='str'). Their length is 4
Numerical features are: Index(['international', 'gpa', 'gmat', 'work_exp'], dtype='str'). Their length is 4


#### Logistic Regression Pipeline Implementation

In [311]:
logistic_regression = LogisticRegression(
    C=1,
    
    # NOTE: The Elastic-Net mixing parameter, 
    # with 0 <= l1_ratio <= 1. Setting l1_ratio=1 gives a pure L1-penalty, 
    # setting l1_ratio=0 a pure L2-penalty. 
    # Any value between 0 and 1 gives an Elastic-Net penalty of the form 
    # l1_ratio * L1 + (1 - l1_ratio) * L2.
    l1_ratio=0.5,

    class_weight="balanced",
    random_state=RANDOM_STATE,
    solver="saga",
    max_iter=1000,
    verbose=False,
    warm_start=False,
    # n_jobs=-1,  NOTE: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. Leave it unspecified
)


num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  

preprocessor = ColumnTransformer([
    ("num", num_pipeline, numerical_features),
    ("cat", cat_pipeline, categorical_features)
])

logistic_regression_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("logregression_classifier", logistic_regression)
])

logistic_regression_pipeline.fit(X_train, y_train)

c:\Users\go_fuck_yourself\Desktop\virtual_env\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('logregression_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the diffe

In [312]:
y_pred = logistic_regression_pipeline.predict(X_test)
logregression_accuracy = accuracy_score(y_test, y_pred)
classifier_metrics[f"{LOGREGRESSION_MODEL_NAME} Classifier"] = {
    "accuracy": logregression_accuracy,
    "no_features": 58,
}
print_basic_metrics(model_name=f"{LOGREGRESSION_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred, include_classification_report=True)

LOGISTIC REGRESSION CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.441
Precision (weighted):    0.831
Recall (weighted):       0.441
F1-Score (weighted):     0.567

Confusion Matrix:
-----------------
True Negatives:  7
False Positives: 10
False Negatives: 68
True Positives:  525

Classification Report:
              precision    recall  f1-score   support

       admit       0.09      0.04      0.05       182
        deny       0.98      0.51      0.67      1038
    waitlist       0.02      0.74      0.04        19

    accuracy                           0.44      1239
   macro avg       0.36      0.43      0.25      1239
weighted avg       0.83      0.44      0.57      1239



### KNN

In [313]:
KNN_MODEL_NAME = "K-Nearest Neighbors"

In [314]:
numerical_features = (
    df
    .drop(columns=["admission"])
    .select_dtypes(include=np.number)
    .columns
)

categorical_features = (
	df
	.drop(columns=["admission"])
	.select_dtypes(include=["string", "object", "str"])
	.columns
)

X = df.drop(columns=["admission"])
y = df["admission"]

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=0.5, random_state=RANDOM_STATE, stratify=df["admission"]
)

In [315]:
X_train

,gender,international,gpa,major,race,gmat,work_exp,work_industry
2268,male,0,3.09,business,white,600.0,4.0,technology
5555,male,1,3.06,business,unknown,590.0,5.0,consulting
1236,male,1,2.96,business,unknown,590.0,6.0,investment banking
3349,female,0,3.22,stem,white,590.0,5.0,energy
2097,female,0,3.14,stem,hispanic,660.0,6.0,investment banking
...,...,...,...,...,...,...,...,...
5247,male,0,3.63,stem,white,780.0,7.0,investment banking
2347,female,0,3.16,humanities,black,650.0,5.0,technology
1280,male,0,3.11,stem,hispanic,660.0,4.0,investment banking
3053,male,0,3.35,humanities,white,690.0,4.0,investment management


In [316]:
X_test

,gender,international,gpa,major,race,gmat,work_exp,work_industry
1054,male,0,3.40,humanities,white,730.0,5.0,technology
1751,male,0,2.86,stem,asian,570.0,5.0,financial services
1936,male,0,3.26,business,asian,690.0,4.0,technology
4400,female,0,3.16,business,hispanic,680.0,5.0,pe/vc
4944,male,0,3.37,business,white,660.0,6.0,pe/vc
...,...,...,...,...,...,...,...,...
1660,male,1,3.31,stem,unknown,760.0,6.0,investment banking
3180,female,0,3.39,business,black,660.0,6.0,consulting
4254,female,0,3.23,humanities,asian,650.0,5.0,other
3780,male,0,3.58,business,asian,760.0,5.0,consulting


In [317]:
y_train.value_counts()

admission
deny        2597
admit        450
waitlist      50
Name: count, dtype: int64

In [318]:
y_test.value_counts()

admission
deny        2597
admit        450
waitlist      50
Name: count, dtype: int64

In [319]:
print(f"Categorical features are: {categorical_features}. Their length is {len(categorical_features)}")
print(f"Numerical features are: {numerical_features}. Their length is {len(numerical_features)}")

Categorical features are: Index(['gender', 'major', 'race', 'work_industry'], dtype='str'). Their length is 4
Numerical features are: Index(['international', 'gpa', 'gmat', 'work_exp'], dtype='str'). Their length is 4


In [320]:
knn_base_classifier = KNeighborsClassifier(
    n_neighbors=5,
    weights="distance",
    metric="minkowski",
    p=2
)

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, numerical_features),
    ("cat", cat_pipeline, categorical_features)
])

knn_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("knn_classifier", KNeighborsClassifier())
])

knn_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('knn_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different trans

In [321]:
y_pred = knn_pipeline.predict(X_test)
knn_accuracy = accuracy_score(y_test, y_pred)
classifier_metrics[f"{KNN_MODEL_NAME} Classifier"] = {
    "accuracy": knn_accuracy,
    "no_features": 58,
}
print_basic_metrics(model_name=f"{KNN_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred, include_classification_report=False)

K-NEAREST NEIGHBORS CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.822
Precision (weighted):    0.773
Recall (weighted):       0.822
F1-Score (weighted):     0.791

Confusion Matrix:
-----------------
True Negatives:  83
False Positives: 367
False Negatives: 133
True Positives:  2464


c:\Users\go_fuck_yourself\Desktop\virtual_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


#### Search for the Best Combination of Hyperparameters for KNN Classifier

In [322]:
param_grid = {
    "knn_classifier__n_neighbors": list(range(3, 25)),
    "knn_classifier__weights": ["uniform", "distance"],
    "knn_classifier__p": [1, 2],
    "knn_classifier__algorithm": ['auto', 'ball_tree', 'kd_tree', 'brute']
}

random_knn = RandomizedSearchCV(
    knn_pipeline,
    param_distributions=param_grid,
    n_iter=5,                   # NOTE: Number of random combinations
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=42
)

random_knn.fit(X_train, y_train)

c:\Users\go_fuck_yourself\Desktop\virtual_env\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [       nan        nan 0.8317682  0.83370629 0.81787847]
  warnings.warn(
c:\Users\go_fuck_yourself\Desktop\virtual_env\Lib\site-packages\sklearn\neighbors\_base.py:591: UserWarning: cannot use tree with sparse input: using brute force
  warnings.warn("cannot use tree with sparse input: using brute force")


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...lassifier())])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'knn_classifier__algorithm': ['auto', 'ball_tree', ...], 'knn_classifier__n_neighbors': [3, 4, ...], 'knn_classifier__p': [1, 2], 'knn_classifier__weights': ['uniform', 'distance']}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",5
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variou

In [323]:
best_knn = random_knn.best_estimator_  
classifier = best_knn.named_steps["knn_classifier"]

print("Best Parameters:", random_knn.best_params_)
print("Best CV Score:", random_knn.best_score_)

y_pred = best_knn.predict(X_test)
tuned_knn_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(model_name=f"Tuned {KNN_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

Best Parameters: {'knn_classifier__weights': 'distance', 'knn_classifier__p': 2, 'knn_classifier__n_neighbors': 19, 'knn_classifier__algorithm': 'ball_tree'}
Best CV Score: 0.8337062900620149
TUNED K-NEAREST NEIGHBORS CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.835
Precision (weighted):    0.770
Recall (weighted):       0.835
F1-Score (weighted):     0.781

Confusion Matrix:
-----------------
True Negatives:  33
False Positives: 416
False Negatives: 45
True Positives:  2552

Classification Report:
              precision    recall  f1-score   support

       admit       0.42      0.07      0.12       450
        deny       0.85      0.98      0.91      2597
    waitlist       0.00      0.00      0.00        50

    accuracy                           0.83      3097
   macro avg       0.42      0.35      0.34      3097
weighted avg       0.77      0.83      0.78      3097



In [324]:
classifier_metrics[f"Tuned {KNN_MODEL_NAME} Classifier"] = {
    "accuracy": tuned_knn_accuracy,
    "no_features": 58
}

pprint.pprint(classifier_metrics)

{'K-Nearest Neighbors Classifier': {'accuracy': 0.8224087826929286,
                                    'no_features': 58},
 'Logistic Regression Classifier': {'accuracy': 0.4406779661016949,
                                    'no_features': 58},
 'Tuned K-Nearest Neighbors Classifier': {'accuracy': 0.8346787213432354,
                                          'no_features': 58}}
